This notebook process our baselines (MolGlue-DB and PROTAC-DB 3.0).

In [ ]:
import pandas as pd
import numpy as np


def clean_compound_name(df, col="Compound_Name"):
    df = df.copy()
    df[col] = df[col].str.replace(r"^cmpd\s+", "", regex=True)
    df[col] = df[col].str.replace(r"^cmp", "", regex=True)
    return df


def standardize_dc50_h(df, col="DC50_h"):
    df = df.copy()
    df[col] = df[col].astype(str).str.replace(r"\s*h\s*$", "", regex=True)
    df[col] = df[col].replace({"nan": np.nan, "": np.nan})
    return df


def clear_orphan_dc50_units(df, dc50_col="DC50", units_col="DC50_units"):
    df = df.copy()
    mask = df[dc50_col].isna() & df[units_col].notna()
    n = int(mask.sum())
    df.loc[mask, units_col] = np.nan
    print(f"Cleared {n} {units_col} values where {dc50_col} was empty")
    return df


def filter_empty_dc50_dmax(
    df, cols=("DC50", "DC50_units", "DC50_h", "Dmax", "Dmax_h", "Dmax_conc")
):
    cols = list(cols)
    mask = df[cols].isna().all(axis=1)
    filtered = df[~mask].copy()
    print(f"Removed {mask.sum()} rows, {len(filtered)} rows remaining")
    return filtered


def add_connectivity_key(df, inchikey_col):
    df = df.copy()
    df["Connectivity_Key"] = df[inchikey_col].str[:14]
    return df


def remove_excluded_dois(df, excluded_csv_path):
    excluded = pd.read_csv(excluded_csv_path)
    excluded_set = set(excluded["DOI"].astype(str).str.strip())

    doi_norm = df["DOI"].astype(str).str.strip()
    mask = doi_norm.isin(excluded_set)
    n_unique_matched = doi_norm[mask].nunique()

    before = len(df)
    filtered = df[~mask].reset_index(drop=True)
    after = len(filtered)

    print(
        f"Removed {before - after} rows ({n_unique_matched} unique excluded DOIs matched)"
    )
    print(f"Shape: {before} -> {after}")
    return filtered


def split_multi_doi_rows(df, doi_col="DOI", sep=";"):
    df = df.copy()
    before = len(df)
    multi_mask = df[doi_col].astype(str).str.contains(sep, na=False)
    n_multi = int(multi_mask.sum())

    df[doi_col] = df[doi_col].astype(str).str.split(sep)
    df = df.explode(doi_col, ignore_index=True)
    df[doi_col] = df[doi_col].str.strip()

    print(f"Split {n_multi} multi-DOI rows; {before} -> {len(df)} rows after expansion")
    return df


def convert_excluded_dois_txt_to_csv(excluded_txt_path, excluded_csv_path):
    with open(excluded_txt_path) as f:
        lines = f.readlines()

    records = []
    current_reason = None
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if line.startswith("#"):
            if "NOT FOUND IN PMC DATABASE" in line:
                current_reason = "NOT_FOUND_IN_PMC_DATABASE"
            elif "IN PMC BUT NOT OPEN ACCESS" in line:
                current_reason = "IN_PMC_BUT_NOT_OPEN_ACCESS"
            continue
        if current_reason is None:
            continue
        for doi in line.split(";"):
            doi = doi.strip()
            if doi:
                records.append({"DOI": doi, "exclude_reason": current_reason})

    excluded_dois_df = pd.DataFrame(records)
    excluded_dois_df.to_csv(excluded_csv_path, index=False)
    print(f"Saved excluded DOIs to: {excluded_csv_path}")
    print(f"Shape: {excluded_dois_df.shape}")
    print(excluded_dois_df["exclude_reason"].value_counts())
    return excluded_dois_df

## PROCESS MOLECULAR GLUES BASELINE (MolGlue-DB)

In [2]:
glue_path = "/Users/yaochenr/project/data/clean_data_csv/glue_data/baseline/26-05-02-baseline-molgluedb-70.csv"

In [3]:
glue_df = pd.read_csv(glue_path)
glue_df = clean_compound_name(glue_df)
glue_df

,Record_ID,Compound_ID,Compound_Name,SMILES,StdInChl,StdInChIKey,Degradation_Target,Recruiter,Recruiter_UniProtID,Cell_Line,Assay,DC50,DC50_units,DC50_h,Dmax,Dmax_h,Dmax_conc,DOI
0,135,129,WBC100,CC(C)[C@@H](N)C(=O)O[C@@H]1[C@@]2(C(C)C)O[C@H]...,InChI=1S/C25H33NO7/c1-10(2)16(26)20(28)30-21-2...,OWLJDBBFRJXPGM-VBIGTWTASA-N,c-Myc,CHIP,A6HD62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.1002/advs.202104344
1,1023,889,D4,O=C1C(C2=CC=CC=C2)=CN(CC2=CC=CC=C2)C(=O)N1CC1=...,InChI=1S/C24H20N2O2/c27-23-22(21-14-8-3-9-15-2...,GWHJFOOJHFJBHW-UHFFFAOYSA-N,NaN,β-TrCP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.1002/advs.202305035
2,1024,890,D2,O=C1C(C2=CC=CC=N2)=CN(C2CC2)C(=O)N1CC1=CC=CC=C1,InChI=1S/C19H17N3O2/c23-18-16(17-8-4-5-11-20-1...,VVICBRKZYCWXJF-UHFFFAOYSA-N,NaN,β-TrCP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.1002/advs.202305035
3,1025,891,D3,O=C1C(C2=CC=CC=N2)=CN(CC2=CC=CC=C2)C(=O)N1CC1=...,InChI=1S/C23H19N3O2/c27-22-20(21-13-7-8-14-24-...,RMULPOPAACYLFW-UHFFFAOYSA-N,NaN,β-TrCP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.1002/advs.202305035
4,248,237,I5,CC1=C(C)C=C(N2N=C(C#N)C(=O)N(C)C2=O)C=C1,InChI=1S/C13H12N4O2/c1-8-4-5-10(6-9(8)2)17-13(...,NBWYKDHVPQBXEU-UHFFFAOYSA-N,NaN,β-TrCP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.1002/advs.202305035
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
706,82,77,HQ019,CC(=O)NC1=NC(CC(=O)NC2=NC=C(C)S2)=CS1,InChI=1S/C11H12N4O2S2/c1-6-4-12-10(19-6)15-9(1...,AVKUTTYOGRAGFM-UHFFFAOYSA-N,NaN,DDB1,Q16531,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.7554/eLife.59994
707,841,774,HQ021,NC(=O)C1=CC=CC=C1NC(=O)CC1=CSC(NC2=NC=CC=C2)=N1,InChI=1S/C17H15N5O2S/c18-16(24)12-5-1-2-6-13(1...,IOFFPPJYBLBCGR-UHFFFAOYSA-N,NaN,DDB1,Q16531,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.7554/eLife.59994
708,843,776,HQ022,NC(=O)CNC(=O)CC1=CSC(NC2=NC=CC=C2)=N1,InChI=1S/C12H13N5O2S/c13-9(18)6-15-11(19)5-8-7...,LNJMZTRTZHJTHB-UHFFFAOYSA-N,NaN,DDB1,Q16531,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.7554/eLife.59994
709,952,831,HQ013,NC1=NC(CC(=O)NC2=NC(C3=CC=CO3)=CS2)=CS1,InChI=1S/C12H10N4O2S2/c13-11-14-7(5-19-11)4-10...,CGQOACVDRKDGKA-UHFFFAOYSA-N,NaN,DDB1,Q16531,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.7554/eLife.59994


### Remove rows with empty DC50 & Dmax

In [4]:
glue_df = clear_orphan_dc50_units(glue_df)
glue_df = filter_empty_dc50_dmax(glue_df)
glue_df.head()

Cleared 0 DC50_units values where DC50 was empty
Removed 575 rows, 136 rows remaining


,Record_ID,Compound_ID,Compound_Name,SMILES,StdInChl,StdInChIKey,Degradation_Target,Recruiter,Recruiter_UniProtID,Cell_Line,Assay,DC50,DC50_units,DC50_h,Dmax,Dmax_h,Dmax_conc,DOI
43,456,417,MG-Lin1,CC1=NN=C2C=CC(C3=CC=CC(N(C)C(=O)/C=C/C4=CC=C(C...,InChI=1S/C23H18F3N5O/c1-15-27-28-21-12-11-20(2...,ROVDMIUDPJJBBI-MDWZMJQESA-N,Lin28,RNF126,Q9BV68,PA-1,NaN,700,nM,NaN,NaN,NaN,NaN,10.1002/mabi.202400427
49,281,266,TMX1,CC1=C(C)C2=C(S1)N1C(C)=NN=C1[C@H](CC(=O)OC(C)(...,InChI=1S/C26H28N4O3S/c1-15-16(2)34-25-22(15)23...,SJISOGWQGIVAGX-DUIUGDAFSA-N,BRD4,DCAF16,Q9NXF7,NaN,NaN,NaN,NaN,NaN,75,16.0,NaN,10.1016/j.ejmech.2024.116904
50,282,267,TMX4128,CC1=C(C)C2=C(S1)N1C(C)=NN=C1[C@H](CC(=O)OC(C)(...,InChI=1S/C26H26N4O3S/c1-15-16(2)34-25-22(15)23...,JEJGRWMOILENAT-FQEVSTJZSA-N,BRD4,DCAF16,Q9NXF7,NaN,NaN,703,nM,NaN,69,16.0,NaN,10.1016/j.ejmech.2024.116904
53,287,269,TMX458,CC1=C(C)C2=C(S1)N1C(C)=NN=C1[C@H](CC(=O)OC(C)(...,InChI=1S/C26H28N4O3S/c1-15-16(2)34-25-22(15)23...,SVDRARONQMNMOU-FQEVSTJZSA-N,BRD4,DCAF16,Q9NXF7,NaN,NaN,> 10,μM,NaN,10,16.0,NaN,10.1016/j.ejmech.2024.116904
55,289,271,MMH249,CC1=C(C)C2=C(S1)N1C(C)=NN=C1[C@H](CC(=O)OC(C)(...,InChI=1S/C25H28ClN5O3S/c1-13-14(2)35-24-21(13)...,UJMDKYUESHTQNZ-SFHVURJKSA-N,BRD4,DCAF16,Q9NXF7,NaN,NaN,8,nM,NaN,NaN,NaN,NaN,10.1016/j.ejmech.2024.116904


### Add Connectivity_Key

In [5]:
glue_df = add_connectivity_key(glue_df, "StdInChIKey")
glue_df[["Compound_Name", "StdInChIKey", "Connectivity_Key"]]

,Compound_Name,StdInChIKey,Connectivity_Key
43,MG-Lin1,ROVDMIUDPJJBBI-MDWZMJQESA-N,ROVDMIUDPJJBBI
49,TMX1,SJISOGWQGIVAGX-DUIUGDAFSA-N,SJISOGWQGIVAGX
50,TMX4128,JEJGRWMOILENAT-FQEVSTJZSA-N,JEJGRWMOILENAT
53,TMX458,SVDRARONQMNMOU-FQEVSTJZSA-N,SVDRARONQMNMOU
55,MMH249,UJMDKYUESHTQNZ-SFHVURJKSA-N,UJMDKYUESHTQNZ
...,...,...,...
699,HQ007,QVRPMHDRWHSFIY-UHFFFAOYSA-N,QVRPMHDRWHSFIY
700,HQ008,GOZYMSUKKKEICN-UHFFFAOYSA-N,GOZYMSUKKKEICN
701,HQ010,IRVKFRZWUBEJNQ-UHFFFAOYSA-N,IRVKFRZWUBEJNQ
702,HQ004,GABXTADXGHTPSI-UHFFFAOYSA-N,GABXTADXGHTPSI


In [6]:
glue_df.to_csv(
    "/Users/yaochenr/project/data/clean_data_csv/glue_data/baseline/26-05-02-glue-baseline-processed.csv",
    index=False,
)

# PROCESS PROTACS BASELINE (PROTAC-DB)

In [7]:
protac_path = (
    "/Users/yaochenr/project/data/clean_data_csv/26-04-24-PROTAC-Baseline-Final.csv"
)

## Split multi-DOI rows

Some baseline rows list two DOIs in the `DOI` field (separated by `;`), e.g. when the same compound is reported in multiple papers. Expand each into one row per DOI so that downstream LLM-vs-baseline comparison can match per-paper.

In [8]:
protac_df = pd.read_csv(protac_path)
protac_df = split_multi_doi_rows(protac_df)
protac_df.head()

Split 9 multi-DOI rows; 9756 -> 9765 rows after expansion


,Record_ID,Compound_ID,Compound_Name,SMILES,InChI,InChIKey,Degradation_Target,Target_Uniprot,Recruiter,Cell_Line,Assay,DC50,DC50_units,DC50_h,Dmax,Dmax_h,Dmax_conc,DOI
0,7623,3745,BD-7148,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5CN(C6=CC=CC7=C6...,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,P25440,CRBN,MV4;11,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520
1,7624,3745,BD-7148,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5CN(C6=CC=CC7=C6...,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,P25440,CRBN,MDA-MB-231,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520
2,7625,3745,BD-7148,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5CN(C6=CC=CC7=C6...,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,P25440,CRBN,MCF-7,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520
3,7625,3745,BD-7148,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5CN(C6=CC=CC7=C6...,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,P25440,CRBN,T47D,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520
4,7647,3747,BD-9136,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5(CCN6CCN(C)CC6)...,InChI=1S/C44H44N10O5S/c1-28-47-48-37-25-59-24-...,WNTMFWJFGDGXOH-UHFFFAOYSA-N,BRD2,P25440,CRBN,MV4;11,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520


## Convert excluded DOIs txt -> CSV

In [9]:
excluded_txt_path = "/Users/yaochenr/project/molecular_glue_extractor/data_source/250825_protac_papers/excluded_dois.txt"
excluded_csv_path = "/Users/yaochenr/project/molecular_glue_extractor/data_source/250825_protac_papers/excluded_dois.csv"

excluded_dois_df = convert_excluded_dois_txt_to_csv(
    excluded_txt_path, excluded_csv_path
)
excluded_dois_df.head()

Saved excluded DOIs to: /Users/yaochenr/project/molecular_glue_extractor/data_source/250825_protac_papers/excluded_dois.csv
Shape: (423, 2)
exclude_reason
NOT_FOUND_IN_PMC_DATABASE     261
IN_PMC_BUT_NOT_OPEN_ACCESS    162
Name: count, dtype: int64


,DOI,exclude_reason
0,10.1021/acs.jmedchem.8b01572,NOT_FOUND_IN_PMC_DATABASE
1,10.1039/c8cc09541h,NOT_FOUND_IN_PMC_DATABASE
2,10.1016/j.chembiol.2023.01.007,NOT_FOUND_IN_PMC_DATABASE
3,10.1039/c9cc08238g,NOT_FOUND_IN_PMC_DATABASE
4,10.1021/acs.jmedchem.1c01774,NOT_FOUND_IN_PMC_DATABASE


## Standardize DC50_h

In [10]:
protac_df = standardize_dc50_h(protac_df)
protac_df[["Compound_Name", "DC50_h"]].head(20)

,Compound_Name,DC50_h
0,BD-7148,4
1,BD-7148,4
2,BD-7148,4
3,BD-7148,4
4,BD-9136,4
5,BD-9136,4
6,BD-9136,4
7,BD-9136,4
8,BD-9136,4
9,BD-9136,4


## Remove rows with empty DC50 & Dmax

In [11]:
protac_df = clear_orphan_dc50_units(protac_df)
protac_df = filter_empty_dc50_dmax(protac_df)
protac_df

Cleared 125 DC50_units values where DC50 was empty
Removed 7546 rows, 2219 rows remaining


,Record_ID,Compound_ID,Compound_Name,SMILES,InChI,InChIKey,Degradation_Target,Target_Uniprot,Recruiter,Cell_Line,Assay,DC50,DC50_units,DC50_h,Dmax,Dmax_h,Dmax_conc,DOI
0,7623,3745,BD-7148,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5CN(C6=CC=CC7=C6...,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,P25440,CRBN,MV4;11,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520
1,7624,3745,BD-7148,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5CN(C6=CC=CC7=C6...,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,P25440,CRBN,MDA-MB-231,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520
2,7625,3745,BD-7148,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5CN(C6=CC=CC7=C6...,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,P25440,CRBN,MCF-7,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520
3,7625,3745,BD-7148,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5CN(C6=CC=CC7=C6...,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,P25440,CRBN,T47D,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520
4,7647,3747,BD-9136,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5(CCN6CCN(C)CC6)...,InChI=1S/C44H44N10O5S/c1-28-47-48-37-25-59-24-...,WNTMFWJFGDGXOH-UHFFFAOYSA-N,BRD2,P25440,CRBN,MV4;11,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2528,8864,5316,NaN,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,InChI=1S/C59H81N11O14S2/c1-36-50(85-34-64-36)4...,PWSQVDFJBKSXJQ-NEFBSMMZSA-N,VHL,P40337,VHL,HeLa,NaN,970,nM,12,NaN,NaN,NaN,10.1002/cbic.202200275
2529,9078,5742,NaN,CC1=C(O)C(=O)C=C2C1=CC=C1[C@@]3(C)CC[C@@]4(C)C...,"InChI=1S/C47H55N3O10/c1-26-27-11-13-33-45(4,29...",ITPPCLZSPZXHER-QQOWXAISSA-N,GRP94,P14625,CRBN,4T1,NaN,980,nM,24,NaN,NaN,NaN,10.1016/j.ejps.2023.106624
2530,3919,3722,NaN,CC1=C(C2=CC=C([C@H](C)NC(=O)[C@@H]3C[C@@H](O)C...,InChI=1S/C52H66N8O7S/c1-34(38-14-16-39(17-15-3...,FLZLBIJRCAAIIQ-MUTBYWARSA-N,NAMPT,P43490,VHL,A2780,NaN,NaN,NaN,24,NaN,NaN,NaN,10.1016/j.bmcl.2023.129393
2541,2097,5990,NaN,COC1=CC=C(/C=C\C2=CC(OC)=C(OC)C(OC)=C2)C=C1OC(...,InChI=1S/C37H39N3O10/c1-46-27-16-14-22(12-13-2...,QXCITHRZQTUQSO-SEYXRHQNSA-N,Alpha-tubulin,NaN,CRBN,A549,NaN,NaN,NaN,48,NaN,NaN,NaN,10.1016/j.ejmech.2023.116067


## Add Connectivity_Key

In [12]:
protac_df = add_connectivity_key(protac_df, "InChIKey")
protac_df[["Compound_Name", "InChIKey", "Connectivity_Key"]]

,Compound_Name,InChIKey,Connectivity_Key
0,BD-7148,JTXHXYXYGGEYBL-UHFFFAOYSA-N,JTXHXYXYGGEYBL
1,BD-7148,JTXHXYXYGGEYBL-UHFFFAOYSA-N,JTXHXYXYGGEYBL
2,BD-7148,JTXHXYXYGGEYBL-UHFFFAOYSA-N,JTXHXYXYGGEYBL
3,BD-7148,JTXHXYXYGGEYBL-UHFFFAOYSA-N,JTXHXYXYGGEYBL
4,BD-9136,WNTMFWJFGDGXOH-UHFFFAOYSA-N,WNTMFWJFGDGXOH
...,...,...,...
2528,NaN,PWSQVDFJBKSXJQ-NEFBSMMZSA-N,PWSQVDFJBKSXJQ
2529,NaN,ITPPCLZSPZXHER-QQOWXAISSA-N,ITPPCLZSPZXHER
2530,NaN,FLZLBIJRCAAIIQ-MUTBYWARSA-N,FLZLBIJRCAAIIQ
2541,NaN,QXCITHRZQTUQSO-SEYXRHQNSA-N,QXCITHRZQTUQSO


## Remove DOIs not avaliable in PMC

In [13]:
protac_df = remove_excluded_dois(protac_df, excluded_csv_path)

Removed 1847 rows (262 unique excluded DOIs matched)
Shape: 2219 -> 372


In [14]:
protac_df.to_csv(
    "/Users/yaochenr/project/data/clean_data_csv/26-05-02-baseline-protacdb-processed-filtered-cleaned.csv",
    index=False,
)